In [2]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    provider="auto",
)

In [2]:
%pip install -q youtube-transcript-api langchain-community faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\HP\AppData\Local\Temp\ipykernel_18640\2340508683.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
video_url = input("Enter the video url")

In [ ]:
from urllib.parse import urlparse, parse_qs
parsed_url = urlparse(video_url)

if parsed_url.hostname in ("youtu.be", "www.youtu.be"):
    video_id = parsed_url.path.lstrip("/")
elif parsed_url.path == "/watch":
    video_id = parse_qs(parsed_url.query).get("v", [None])[0]
elif parsed_url.path.startswith(("/embed/", "/shorts/")):
    video_id = parsed_url.path.split("/")[2]
else:
    video_id = None

In [ ]:
try:
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)
except TranscriptsDisabled:
    print("No captions available for this video")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.split_documents([transcript])

In [6]:
%pip install -q sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\HP\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

c:\Users\HP\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

OSError: Can't load the model for 'sentence-transformers/all-MiniLM-L6-v2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'sentence-transformers/all-MiniLM-L6-v2' is the correct path to a directory containing a file named pytorch_model.bin.

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embeddings=embeddings
)

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
        You are an Assistant.
        Answer the questions only from the provided transcript context
        If you find the context to be insufficient, just say that you don't know.
        {context}
        Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [3]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.ouptut_parsers import StrOutputParser

ModuleNotFoundError: No module named 'langchain_core.ouptut_parsers'

In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'qeustion': RunnablePassthrough
})

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1922739158.py, line 2)

In [ ]:
qeury = input("Enter your query")

In [ ]:
parallel_chain.invoke(qeury)

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
user_query = input("Enter your question:")
main_chain.invoke(user_query)